# AU-FISH Multi-Seed Individual Run

**Protocol:** `source_based`  
**Model:** `YOLO11m`  
**Seed:** `2026`  
**Experiment tag:** `multiseed_fullset_dedup_v1`

Notebook ini hanya menangani satu training run. Semua output disimpan pada folder
protocol, model, dan seed yang spesifik sehingga tidak bercampur dengan run lain.

Split dataset tidak diubah antar-seed. Preprocessing final tetap menggunakan:

- explicit image lists;
- filtered two-class annotations: coral and fish;
- exact duplicate removal yang telah dilakukan sebelumnya;
- minimum box width/height 5 px;
- minimum box area 25 px²;
- common filtered COCO ground truth.

Jangan mengubah protocol, model, seed, split, atau output root di tengah eksekusi.

In [1]:
# CELL 1 — INSTALL PINNED DEPENDENCIES
!pip -q install ultralytics==8.4.102 pycocotools pandas pyyaml pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.3 MB/s eta 0:00:00


In [2]:
# CELL 2 — MOUNT GOOGLE DRIVE
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# CELL 3 — FIXED RUN CONFIGURATION
import contextlib
import hashlib
import io
import json
import os
import platform
import random
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from ultralytics import YOLO
MODEL_CLASS = YOLO

PROTOCOL = "source_based"
MODEL_KEY = "yolo11m"
MODEL_LABEL = "YOLO11m"
SEED = 2026

PROJECT_ROOT = Path("/content/drive/MyDrive/au-fish/dataset_v2")
DATASET_ROOT = PROJECT_ROOT / PROTOCOL

INDIVIDUAL_TAG = "multiseed_fullset_dedup_v1"
LEGACY_FINAL_TAG = "simple_wbf_fullset_dedup_v1"

YAML_PATH = DATASET_ROOT / f"aufish_{PROTOCOL}_fullset.yaml"

TRAIN_RUN_ROOT = (
    DATASET_ROOT
    / "runs"
    / INDIVIDUAL_TAG
    / MODEL_KEY
    / f"seed_{SEED}"
)

EVAL_RUN_ROOT = (
    DATASET_ROOT
    / "runs_eval"
    / INDIVIDUAL_TAG
    / MODEL_KEY
    / f"seed_{SEED}"
)

EXPERIMENT_ROOT = (
    DATASET_ROOT
    / "experiment_logs"
    / INDIVIDUAL_TAG
)

RUN_ROOT = (
    EXPERIMENT_ROOT
    / "individual"
    / MODEL_KEY
    / f"seed_{SEED}"
)

RAW_DIR = RUN_ROOT / "raw_predictions"
RESULTS_DIR = RUN_ROOT / "results"
MANIFEST_DIR = RUN_ROOT / "manifests"
SHARED_GT_DIR = EXPERIMENT_ROOT / "shared_ground_truth"

for path in [
    TRAIN_RUN_ROOT,
    EVAL_RUN_ROOT,
    RAW_DIR,
    RESULTS_DIR,
    MANIFEST_DIR,
    SHARED_GT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

PRETRAINED_WEIGHTS = "yolo11m.pt"
EPOCHS = 100
IMAGE_SIZE = 640
TRAIN_BATCH = 16
PREDICT_BATCH = 8
DEVICE = 0 if torch.cuda.is_available() else "cpu"

TRAIN_RUN_NAME = "train"
TRAIN_OUTPUT_DIR = TRAIN_RUN_ROOT / TRAIN_RUN_NAME
BEST_CHECKPOINT = TRAIN_OUTPUT_DIR / "weights" / "best.pt"

FORCE_RETRAIN = False
FORCE_REINFERENCE = False

EXPECTED_IMAGES = {
    "time_based": {"train": 1061, "val": 203, "test": 282},
    "source_based": {"train": 1083, "val": 232, "test": 231},
}

print("=" * 80)
print("AU-FISH MULTI-SEED INDIVIDUAL RUN")
print("Protocol :", PROTOCOL)
print("Model    :", MODEL_LABEL)
print("Seed     :", SEED)
print("Run root :", RUN_ROOT)
print("GPU      :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("=" * 80)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
AU-FISH MULTI-SEED INDIVIDUAL RUN
Protocol : source_based
Model    : YOLO11m
Seed     : 2026
Run root : /content/drive/MyDrive/au-fish/dataset_v2/source_based/experiment_logs/multiseed_fullset_dedup_v1/individual/yolo11m/seed_2026
GPU      : NVIDIA L4


In [4]:
# CELL 4 — REPRODUCIBILITY AND FILE HELPERS
def set_global_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_suffix(path.suffix + ".tmp")

    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

    temporary.replace(path)


def save_run_config():
    config = {
        "protocol": PROTOCOL,
        "model": MODEL_KEY,
        "seed": SEED,
        "experiment_tag": INDIVIDUAL_TAG,
        "dataset_yaml": str(YAML_PATH),
        "pretrained_weights": PRETRAINED_WEIGHTS,
        "epochs": EPOCHS,
        "image_size": IMAGE_SIZE,
        "train_batch": TRAIN_BATCH,
        "predict_batch": PREDICT_BATCH,
        "deterministic": True,
        "raw_prediction_confidence": 0.001,
        "raw_prediction_iou": 0.70,
        "raw_prediction_max_det": 300,
        "test_time_augmentation": False,
        "checkpoint_selection": "Ultralytics validation fitness",
    }

    save_json(
        config,
        MANIFEST_DIR / "run_config.json",
    )


set_global_seed(SEED)
save_run_config()
print("Global seed set:", SEED)

Global seed set: 2026


In [5]:
# CELL 5 — DATASET AND SHARED GROUND-TRUTH PREFLIGHT
if not YAML_PATH.exists():
    raise FileNotFoundError(
        f"Final explicit-list YAML not found: {YAML_PATH}"
    )

with open(YAML_PATH, "r", encoding="utf-8") as file:
    data_yaml = yaml.safe_load(file)

required_yaml_keys = {"train", "val", "test", "names"}

if not required_yaml_keys.issubset(data_yaml):
    raise RuntimeError(
        f"YAML missing keys: {required_yaml_keys - set(data_yaml)}"
    )

for split_name in ["train", "val", "test"]:
    list_path = Path(data_yaml[split_name])

    if not list_path.exists():
        raise FileNotFoundError(
            f"Explicit image list not found: {list_path}"
        )

    image_paths = [
        line.strip()
        for line in list_path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

    expected = EXPECTED_IMAGES[PROTOCOL][split_name]

    if len(image_paths) != expected:
        raise RuntimeError(
            f"{split_name}: expected {expected} images, "
            f"found {len(image_paths)}"
        )

    if len(image_paths) != len(set(image_paths)):
        raise RuntimeError(
            f"{split_name}: duplicate paths found"
        )

    missing = [
        path
        for path in image_paths
        if not Path(path).exists()
    ]

    if missing:
        raise FileNotFoundError(
            f"{split_name}: {len(missing)} image files missing. "
            f"First: {missing[0]}"
        )

    print(
        f"✅ {split_name}: {len(image_paths)} explicit paths"
    )

legacy_gt_dir = (
    DATASET_ROOT
    / "experiment_logs"
    / LEGACY_FINAL_TAG
    / "shared_ground_truth"
)

gt_files = {
    "validation": "val_gt_filtered.json",
    "test": "test_gt_filtered.json",
}

for split_label, file_name in gt_files.items():
    source_path = legacy_gt_dir / file_name
    target_path = SHARED_GT_DIR / file_name

    if not source_path.exists():
        raise FileNotFoundError(
            f"Locked common GT not found: {source_path}"
        )

    if not target_path.exists():
        shutil.copy2(source_path, target_path)

    gt = load_json(target_path)
    expected = EXPECTED_IMAGES[PROTOCOL][
        "val" if split_label == "validation" else "test"
    ]

    if len(gt["images"]) != expected:
        raise RuntimeError(
            f"{split_label} GT image count mismatch: "
            f"{len(gt['images'])} != {expected}"
        )

    print(
        f"✅ {split_label} GT: "
        f"{len(gt['images'])} images, "
        f"{len(gt['annotations'])} annotations"
    )

✅ train: 1083 explicit paths
✅ val: 232 explicit paths
✅ test: 231 explicit paths
✅ validation GT: 232 images, 1857 annotations
✅ test GT: 231 images, 1820 annotations


In [6]:
# CELL 6 — TRAIN YOLO11M
if BEST_CHECKPOINT.exists() and not FORCE_RETRAIN:
    print("Checkpoint already exists. Training skipped:")
    print(BEST_CHECKPOINT)
else:
    if TRAIN_OUTPUT_DIR.exists() and FORCE_RETRAIN:
        shutil.rmtree(TRAIN_OUTPUT_DIR)

    model = MODEL_CLASS(PRETRAINED_WEIGHTS)

    train_result = model.train(
        data=str(YAML_PATH),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=TRAIN_BATCH,
        device=DEVICE,
        seed=SEED,
        deterministic=True,
        project=str(TRAIN_RUN_ROOT),
        name=TRAIN_RUN_NAME,
        exist_ok=False,
        resume=False,
        verbose=True,
    )

if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Best checkpoint not produced: {BEST_CHECKPOINT}"
    )

print("✅ Best checkpoint:", BEST_CHECKPOINT)
print("SHA256:", sha256_file(BEST_CHECKPOINT))

New https://pypi.org/project/ultralytics/8.4.108 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/au-fish/dataset_v2/source_based/aufish_source_based_fullset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, m

In [7]:
# ============================================================
# RECOVER ACTUAL ULTRALYTICS CHECKPOINT
# ============================================================
from pathlib import Path

checkpoint_candidates = list(
    TRAIN_RUN_ROOT.glob("train*/weights/best.pt")
)

checkpoint_candidates = [
    path
    for path in checkpoint_candidates
    if path.exists()
]

if not checkpoint_candidates:
    raise FileNotFoundError(
        f"Tidak ada best.pt di bawah: {TRAIN_RUN_ROOT}"
    )

# Ambil checkpoint dari run yang paling baru dimodifikasi.
checkpoint_candidates.sort(
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

BEST_CHECKPOINT = checkpoint_candidates[0]

print("✅ Actual checkpoint found:")
print(BEST_CHECKPOINT)
print("Run folder:", BEST_CHECKPOINT.parents[1])

✅ Actual checkpoint found:
/content/drive/MyDrive/au-fish/dataset_v2/source_based/runs/multiseed_fullset_dedup_v1/yolo11m/seed_2026/train/weights/best.pt
Run folder: /content/drive/MyDrive/au-fish/dataset_v2/source_based/runs/multiseed_fullset_dedup_v1/yolo11m/seed_2026/train


In [8]:
# CELL 7 — FRAMEWORK-NATIVE VALIDATION AND TEST DIAGNOSTICS
diagnostic_rows = []

trained_model = MODEL_CLASS(str(BEST_CHECKPOINT))

for split_name in ["val", "test"]:
    output_name = f"{split_name}_diagnostic"

    metrics = trained_model.val(
        data=str(YAML_PATH),
        split=split_name,
        imgsz=IMAGE_SIZE,
        batch=PREDICT_BATCH,
        device=DEVICE,
        project=str(EVAL_RUN_ROOT),
        name=output_name,
        exist_ok=True,
        verbose=True,
    )

    row = {
        "protocol": PROTOCOL,
        "model": MODEL_KEY,
        "seed": SEED,
        "split": split_name,
        "mAP50_95": float(metrics.box.map),
        "mAP50": float(metrics.box.map50),
        "mAP75": float(metrics.box.map75),
        "AP_coral": float(metrics.box.maps[0]),
        "AP_fish": float(metrics.box.maps[1]),
    }

    diagnostic_rows.append(row)

diagnostic_df = pd.DataFrame(diagnostic_rows)
diagnostic_df.to_csv(
    RESULTS_DIR / "framework_native_results.csv",
    index=False,
)

display(diagnostic_df)

Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO11m summary (fused): 126 layers, 20,031,574 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.0 ms, read: 722.5±288.8 MB/s, size: 3789.3 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/1kW1zeaSZ1KWylT60LjX9VMVZYNBF137Z/au-fish/dataset_v2/source_based/val/labels.cache... 232 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 232/232 57.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 5.9it/s 4.9s
                   all        232       1857      0.557      0.418      0.428      0.258
                 coral        162        288      0.515      0.601      0.564      0.374
                  fish        107       1569      0.599      0.235      0.291      0.142
Speed: 0.7ms preprocess, 5.8ms inference, 0.0ms loss, 4.8ms postprocess per image
Results saved to /content/drive/.shortcut-targets-b

,protocol,model,seed,split,mAP50_95,mAP50,mAP75,AP_coral,AP_fish
0,source_based,yolo11m,2026,val,0.257963,0.427963,0.273520,0.373922,0.142004
1,source_based,yolo11m,2026,test,0.176739,0.341697,0.163303,0.192638,0.160840


In [9]:
# CELL 8 — RAW PREDICTION AND COMMON COCOEVAL HELPERS
def blank_record(image_info):
    return {
        "image_id": int(image_info["id"]),
        "file_name": str(image_info["file_name"]),
        "width": int(image_info["width"]),
        "height": int(image_info["height"]),
        "boxes": [],
        "scores": [],
        "labels": [],
    }


def gt_path_for(split_label):
    file_name = (
        "val_gt_filtered.json"
        if split_label == "validation"
        else "test_gt_filtered.json"
    )

    return SHARED_GT_DIR / file_name


def split_dir_name(split_label):
    return "val" if split_label == "validation" else "test"


def build_ordered_image_paths(gt):
    split_dir = split_dir_name(CURRENT_SPLIT)

    paths = []

    for image_info in gt["images"]:
        path = (
            DATASET_ROOT
            / split_dir
            / "images"
            / image_info["file_name"]
        )

        if not path.exists():
            raise FileNotFoundError(
                f"Image missing: {path}"
            )

        paths.append(str(path))

    return paths


def run_raw_inference(model, split_label):
    global CURRENT_SPLIT
    CURRENT_SPLIT = split_label

    gt = load_json(gt_path_for(split_label))
    image_infos = gt["images"]
    image_paths = build_ordered_image_paths(gt)

    output_path = (
        RAW_DIR
        / f"{split_label}_{MODEL_KEY}.json"
    )

    if output_path.exists() and not FORCE_REINFERENCE:
        raw = load_json(output_path)

        if len(raw) != len(image_infos):
            raise RuntimeError(
                f"Existing raw prediction coverage mismatch: "
                f"{len(raw)} != {len(image_infos)}"
            )

        print("Raw prediction already exists:", output_path)
        return raw

    results = model.predict(
        source=image_paths,
        conf=0.001,
        iou=0.70,
        max_det=300,
        imgsz=IMAGE_SIZE,
        batch=PREDICT_BATCH,
        device=DEVICE,
        augment=False,
        save=False,
        stream=False,
        verbose=False,
    )

    if len(results) != len(image_infos):
        raise RuntimeError(
            f"Prediction count mismatch: "
            f"{len(results)} != {len(image_infos)}"
        )

    raw = {}

    # Important: map by validated input order, not result.path.
    for image_info, result in zip(image_infos, results):
        record = blank_record(image_info)

        if result.boxes is not None:
            boxes = (
                result.boxes.xyxy
                .detach()
                .cpu()
                .numpy()
            )
            scores = (
                result.boxes.conf
                .detach()
                .cpu()
                .numpy()
            )
            labels = (
                result.boxes.cls
                .detach()
                .cpu()
                .numpy()
            )

            for box, score, label in zip(
                boxes,
                scores,
                labels,
            ):
                x1, y1, x2, y2 = map(float, box)

                if x2 <= x1 or y2 <= y1:
                    continue

                record["boxes"].append(
                    [x1, y1, x2, y2]
                )
                record["scores"].append(
                    float(score)
                )
                record["labels"].append(
                    int(label)
                )

        raw[str(record["image_id"])] = record

    save_json(raw, output_path)

    print(
        f"✅ {split_label} raw predictions: "
        f"{len(raw)} images -> {output_path}"
    )

    return raw


def raw_to_coco_rows(raw):
    rows = []

    for record in raw.values():
        for box, score, label in zip(
            record["boxes"],
            record["scores"],
            record["labels"],
        ):
            x1, y1, x2, y2 = map(float, box)
            width = x2 - x1
            height = y2 - y1

            if width <= 0 or height <= 0:
                continue

            rows.append({
                "image_id": int(record["image_id"]),
                "category_id": int(label) + 1,
                "bbox": [x1, y1, width, height],
                "score": float(score),
            })

    return rows


def evaluate_common_coco(gt_path, raw):
    rows = raw_to_coco_rows(raw)

    if not rows:
        raise RuntimeError("No predictions for COCOeval.")

    with contextlib.redirect_stdout(io.StringIO()):
        coco_gt = COCO(str(gt_path))
        coco_dt = coco_gt.loadRes(rows)

        evaluator = COCOeval(
            coco_gt,
            coco_dt,
            "bbox",
        )
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    class_ap = {}

    for category_id, class_name in [
        (1, "coral"),
        (2, "fish"),
    ]:
        with contextlib.redirect_stdout(io.StringIO()):
            class_eval = COCOeval(
                coco_gt,
                coco_dt,
                "bbox",
            )
            class_eval.params.catIds = [
                category_id
            ]
            class_eval.evaluate()
            class_eval.accumulate()
            class_eval.summarize()

        class_ap[class_name] = float(
            class_eval.stats[0]
        )

    return {
        "mAP50_95": float(evaluator.stats[0]),
        "mAP50": float(evaluator.stats[1]),
        "mAP75": float(evaluator.stats[2]),
        "AP_coral": class_ap["coral"],
        "AP_fish": class_ap["fish"],
        "worst_class_AP": min(
            class_ap["coral"],
            class_ap["fish"],
        ),
        "detections": len(rows),
    }

In [10]:
# CELL 9 — GENERATE VALIDATION AND TEST RAW PREDICTIONS
inference_model = MODEL_CLASS(
    str(BEST_CHECKPOINT)
)

common_rows = []

for split_label in [
    "validation",
    "test",
]:
    raw = run_raw_inference(
        inference_model,
        split_label,
    )

    metrics = evaluate_common_coco(
        gt_path_for(split_label),
        raw,
    )

    common_rows.append({
        "protocol": PROTOCOL,
        "model": MODEL_KEY,
        "seed": SEED,
        "split": split_label,
        **metrics,
    })

common_df = pd.DataFrame(common_rows)
common_df.to_csv(
    RESULTS_DIR / "common_cocoeval_results.csv",
    index=False,
)

display(common_df)

✅ validation raw predictions: 232 images -> /content/drive/MyDrive/au-fish/dataset_v2/source_based/experiment_logs/multiseed_fullset_dedup_v1/individual/yolo11m/seed_2026/raw_predictions/validation_yolo11m.json
✅ test raw predictions: 231 images -> /content/drive/MyDrive/au-fish/dataset_v2/source_based/experiment_logs/multiseed_fullset_dedup_v1/individual/yolo11m/seed_2026/raw_predictions/test_yolo11m.json


,protocol,model,seed,split,mAP50_95,mAP50,mAP75,AP_coral,AP_fish,worst_class_AP,detections
0,source_based,yolo11m,2026,validation,0.250107,0.403740,0.269087,0.362786,0.137428,0.137428,21229
1,source_based,yolo11m,2026,test,0.174190,0.336665,0.165681,0.182373,0.166006,0.166006,30480


In [11]:
# CELL 10 — CHECKPOINT AND OUTPUT MANIFEST
manifest = {
    "protocol": PROTOCOL,
    "model": MODEL_KEY,
    "seed": SEED,
    "checkpoint_path": str(BEST_CHECKPOINT),
    "checkpoint_sha256": sha256_file(
        BEST_CHECKPOINT
    ),
    "dataset_yaml": str(YAML_PATH),
    "validation_raw_predictions": str(
        RAW_DIR
        / f"validation_{MODEL_KEY}.json"
    ),
    "test_raw_predictions": str(
        RAW_DIR
        / f"test_{MODEL_KEY}.json"
    ),
    "framework_native_results": str(
        RESULTS_DIR
        / "framework_native_results.csv"
    ),
    "common_cocoeval_results": str(
        RESULTS_DIR
        / "common_cocoeval_results.csv"
    ),
}

save_json(
    manifest,
    MANIFEST_DIR / "checkpoint_manifest.json",
)

required_outputs = [
    Path(manifest["checkpoint_path"]),
    Path(manifest["validation_raw_predictions"]),
    Path(manifest["test_raw_predictions"]),
    Path(manifest["framework_native_results"]),
    Path(manifest["common_cocoeval_results"]),
]

missing = [
    str(path)
    for path in required_outputs
    if not path.exists()
]

if missing:
    raise RuntimeError(
        "Missing final outputs:\n"
        + "\n".join(missing)
    )

print("✅ Individual multi-seed run complete.")
print("Run root:", RUN_ROOT)
print("Outputs are isolated by protocol/model/seed.")

✅ Individual multi-seed run complete.
Run root: /content/drive/MyDrive/au-fish/dataset_v2/source_based/experiment_logs/multiseed_fullset_dedup_v1/individual/yolo11m/seed_2026
Outputs are isolated by protocol/model/seed.


## Output structure

```text
{protocol}/
├── runs/multiseed_fullset_dedup_v1/{model}/seed_{seed}/train/
├── runs_eval/multiseed_fullset_dedup_v1/{model}/seed_{seed}/
└── experiment_logs/multiseed_fullset_dedup_v1/
    └── individual/{model}/seed_{seed}/
        ├── raw_predictions/
        ├── results/
        └── manifests/
```

Jangan memindahkan raw predictions ke folder seed lain. Pairwise WBF akan
membaca file berdasarkan protocol, model, dan seed yang sama.